# Практическое занятие №3: Свёртка и корреляционный анализ сигналов

## 1. Функции Python для свёртки и корреляции

### 1.1. `np.convolve(x, h, mode)`

- `mode='full'` – полная линейная свёртка (длина $N_x+N_h-1$).
- `mode='same'` – центральная часть той же длины, что и $x$.
- `mode='valid'` – только те точки, где ядро полностью перекрывается с сигналом.

### 1.2. `np.correlate(x, y, mode)`

Аналогичные режимы. При `mode='full'` результат имеет длину $N_x+N_y-1$. Индекс 0 соответствует полному перекрытию сигналов.

Для получения сдвига, при котором достигается максимум корреляции, используется формула:
```python
corr = np.correlate(x, y, mode='full')
delay = np.argmax(corr) - (len(x) - 1)
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.signal.windows import gaussian
from scipy.io import wavfile

np.random.seed(42)
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.grid'] = True


## 2. Гауссовы окна и их свёртка

Гауссово окно можно сгенерировать с помощью `scipy.signal.windows.gaussian(M, std)`.

**Аналитическое свойство:** свёртка двух гауссовых функций с дисперсиями $\sigma_1^2$ и $\sigma_2^2$ даёт гауссову функцию с дисперсией $\sigma_1^2+\sigma_2^2$:
$$
(f_{\sigma_1} * f_{\sigma_2})(t) = f_{\sqrt{\sigma_1^2+\sigma_2^2}}(t).
$$
Это свойство используется в задании 1 для сравнения численного результата с теоретическим.


In [ ]:
M1, std1 = 101, 8
M2, std2 = 121, 12

g1 = gaussian(M1, std=std1)
g2 = gaussian(M2, std=std2)
g1 = g1 / np.sum(g1)
g2 = g2 / np.sum(g2)

conv_g = np.convolve(g1, g2, mode='full')

sigma_theory = np.sqrt(std1**2 + std2**2)

n1 = np.arange(-(M1 // 2), M1 // 2 + 1)
n2 = np.arange(-(M2 // 2), M2 // 2 + 1)
n_conv = np.arange(n1[0] + n2[0], n1[-1] + n2[-1] + 1)

g_theory = np.exp(-(n_conv**2) / (2 * sigma_theory**2))
g_theory = g_theory / np.sum(g_theory)

mse = np.mean((conv_g - g_theory) ** 2)
max_abs_err = np.max(np.abs(conv_g - g_theory))

print(f'Теоретическая sigma: {sigma_theory:.4f}')
print(f'MSE между численной и теоретической кривой: {mse:.2e}')
print(f'MAX |ошибка|: {max_abs_err:.2e}')

plt.figure()
plt.plot(n_conv, conv_g, label='Свёртка g1 * g2', lw=2)
plt.plot(n_conv, g_theory, '--', label='Теоретическая гауссиана', lw=2)
plt.title('Свёртка двух гауссовых окон')
plt.xlabel('n')
plt.ylabel('Амплитуда')
plt.legend()
plt.show()


**Ответ по разделу 2:** свёртка двух нормированных гауссовых окон совпадает с теоретической гауссианой с $\sigma=\sqrt{\sigma_1^2+\sigma_2^2}$ (численная ошибка мала).


## 3. Фильтрация с помощью свёртки

**Прямоугольное окно (усреднение):**
```python
h_rect = signal.windows.boxcar(L)/L
```
Простое скользящее среднее.

**Гауссовское окно:**
```python
h_gauss = signal.windows.gaussian(L, std)
h_gauss = h_gauss / np.sum(h_gauss)  # нормализация
```


In [ ]:
N = 1500
t = np.linspace(0, 1, N, endpoint=False)

clean = np.sin(2 * np.pi * 5 * t) + 0.35 * np.sin(2 * np.pi * 17 * t + 0.4)
noisy = clean + 0.5 * np.random.randn(N)

L = 31
h_rect = signal.windows.boxcar(L) / L

h_gauss = gaussian(L, std=5)
h_gauss = h_gauss / np.sum(h_gauss)

filtered_rect = np.convolve(noisy, h_rect, mode='same')
filtered_gauss = np.convolve(noisy, h_gauss, mode='same')

def snr_db(reference, estimate):
    err = reference - estimate
    return 10 * np.log10(np.sum(reference**2) / np.sum(err**2))

snr_before = snr_db(clean, noisy)
snr_rect = snr_db(clean, filtered_rect)
snr_gauss = snr_db(clean, filtered_gauss)

print(f'SNR до фильтрации: {snr_before:.2f} dB')
print(f'SNR после boxcar: {snr_rect:.2f} dB')
print(f'SNR после gaussian: {snr_gauss:.2f} dB')

plt.figure(figsize=(12, 8))

plt.subplot(3, 1, 1)
plt.plot(t, clean, label='Исходный чистый сигнал', lw=1.5)
plt.plot(t, noisy, label='Шумный сигнал', alpha=0.6)
plt.title('Сигналы до фильтрации')
plt.xlabel('Время, с')
plt.ylabel('Амплитуда')
plt.legend()

plt.subplot(3, 1, 2)
plt.plot(t, clean, label='Чистый сигнал', lw=1.5)
plt.plot(t, filtered_rect, label='После boxcar')
plt.title('Фильтрация прямоугольным окном')
plt.xlabel('Время, с')
plt.ylabel('Амплитуда')
plt.legend()

plt.subplot(3, 1, 3)
plt.plot(t, clean, label='Чистый сигнал', lw=1.5)
plt.plot(t, filtered_gauss, label='После gaussian')
plt.title('Фильтрация гауссовым окном')
plt.xlabel('Время, с')
plt.ylabel('Амплитуда')
plt.legend()

plt.tight_layout()
plt.show()


**Ответ по разделу 3:** оба фильтра улучшают SNR; гауссов фильтр даёт более мягкое сглаживание и обычно меньше искажает форму сигнала на переходах.


## 4. Поиск временной задержки с помощью кросс-корреляции

**Идея:** если $y[n] = x[n-d] + \text{шум}$, то кросс-корреляция $R_{xy}[m]$ имеет максимум при $m = d$.

**Алгоритм:**
1. Вычислить `corr = np.correlate(x, y, mode='full')`.
2. Найти индекс максимума: `idx = np.argmax(corr)`.
3. Задержка = `idx - (len(x)-1)`.


In [ ]:
N = 1000
n = np.arange(N)

x = np.sin(2 * np.pi * 0.03 * n) + 0.7 * np.sin(2 * np.pi * 0.09 * n + 0.5)

true_delay = 37
y = np.zeros_like(x)
y[true_delay:] = x[:-true_delay]
y_noisy = y + 0.25 * np.random.randn(N)

corr = np.correlate(y_noisy, x, mode='full')
estimated_delay = np.argmax(corr) - (len(x) - 1)

lags = np.arange(-(len(x) - 1), len(y_noisy))

print(f'Истинная задержка: {true_delay} отсчётов')
print(f'Оценённая задержка: {estimated_delay} отсчётов')

plt.figure(figsize=(12, 8))

plt.subplot(2, 1, 1)
plt.plot(x[:250], label='x[n] (опорный)')
plt.plot(y_noisy[:250], label='y[n] (задержан + шум)', alpha=0.8)
plt.title('Сигналы для оценки задержки')
plt.xlabel('n')
plt.ylabel('Амплитуда')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(lags, corr, label='Кросс-корреляция')
plt.axvline(estimated_delay, color='r', linestyle='--', label=f'Оценка: {estimated_delay}')
plt.axvline(true_delay, color='g', linestyle=':', label=f'Истина: {true_delay}')
plt.title('Оценка задержки по максимуму корреляции')
plt.xlabel('Лаг (отсчёты)')
plt.ylabel('Rxy')
plt.legend()

plt.tight_layout()
plt.show()


**Ответ по разделу 4:** задержка корректно оценивается как `argmax(corr) - (len(x)-1)`; в примере оценённая задержка совпала с истинной.


## 5. Обнаружение шаблона в зашумлённом сигнале

**Задача:** найти в длинном сигнале позицию, где содержится известный короткий шаблон.

**Метод:**
- Вычислить кросс-корреляцию между длинным сигналом и шаблоном (удобно использовать `mode='valid'`, чтобы избежать краевых эффектов).
- Пик корреляции указывает на позицию совпадения.


In [ ]:
N_long = 4000
template_len = 180
n_temp = np.arange(template_len)

template = np.sin(2 * np.pi * 0.06 * n_temp) * signal.windows.gaussian(template_len, std=template_len / 7)
template = template / np.linalg.norm(template)

long_signal = 0.35 * np.random.randn(N_long)
true_pos = 2350
long_signal[true_pos:true_pos + template_len] += 2.2 * template

corr_valid = np.correlate(long_signal, template, mode='valid')
estimated_pos = int(np.argmax(corr_valid))

print(f'Истинная позиция шаблона: {true_pos}')
print(f'Оценённая позиция: {estimated_pos}')
print(f'Ошибка: {estimated_pos - true_pos} отсчётов')

plt.figure(figsize=(12, 8))

plt.subplot(2, 1, 1)
plt.plot(long_signal, label='Длинный сигнал')
plt.axvline(true_pos, color='g', linestyle=':', label=f'Истина: {true_pos}')
plt.axvline(estimated_pos, color='r', linestyle='--', label=f'Оценка: {estimated_pos}')
plt.title('Длинный сигнал с встроенным шаблоном')
plt.xlabel('n')
plt.ylabel('Амплитуда')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(corr_valid, label="Корреляция (mode='valid')")
plt.axvline(estimated_pos, color='r', linestyle='--', label=f'Пик: {estimated_pos}')
plt.title('Поиск позиции шаблона по корреляции')
plt.xlabel('Позиция начала шаблона')
plt.ylabel('Корреляция')
plt.legend()

plt.tight_layout()
plt.show()


**Ответ по разделу 5:** при `mode='valid'` пик кросс-корреляции точно указывает позицию начала шаблона в длинном зашумлённом сигнале.


## 6. Работа с аудио в Python

**Чтение WAV-файла:**
```python
from scipy.io import wavfile
rate, data = wavfile.read('file.wav')
```
- `rate` – частота дискретизации (Гц).
- `data` – массив целых чисел. Для стерео форма `(N, 2)`.

**Нормализация в диапазон [-1, 1] (для 16-битного аудио):**
```python
data = data / 32767.0
```

**Прослушивание в Jupyter/Colab:**
```python
from IPython.display import Audio
Audio(data, rate=rate)
```

**Поиск фрагмента:**
- Вычислить кросс-корреляцию между полным сигналом и фрагментом.
- Найти индекс максимума.
- Вырезать соответствующий участок и убедиться в совпадении (визуально и на слух).


In [ ]:
try:
    from IPython.display import Audio, display
    ipython_available = True
except ImportError:
    ipython_available = False

!wget -q -O sample-3s.wav https://raw.githubusercontent.com/ItserX/dsp-seminars/main/data/sample-3s.wav

audio_path = 'sample-3s.wav'
rate, data = wavfile.read(audio_path)
print(f'Используется файл: {audio_path}')

if data.ndim == 2:
    data_mono = data.mean(axis=1)
else:
    data_mono = data.astype(np.float64)

if np.issubdtype(data.dtype, np.integer):
    data_mono = data_mono / np.iinfo(data.dtype).max
else:
    data_mono = data_mono.astype(np.float64)

frag_start_true = int(0.95 * rate)
frag_duration_sec = 0.08
frag_len = int(frag_duration_sec * rate)

fragment = data_mono[frag_start_true:frag_start_true + frag_len].copy()
fragment_noisy = fragment + 0.002 * np.random.randn(frag_len)

corr_audio = signal.correlate(data_mono, fragment_noisy, mode='valid', method='fft')
est_start = int(np.argmax(corr_audio))
est_end = est_start + frag_len

err_samples = est_start - frag_start_true
err_ms = 1000 * err_samples / rate

print(f'Частота дискретизации: {rate} Гц')
print(f'Длина записи: {len(data_mono)} отсчётов ({len(data_mono)/rate:.2f} с)')
print(f'Истинный старт фрагмента: {frag_start_true}')
print(f'Оценённый старт: {est_start}')
print(f'Ошибка: {err_samples} отсчётов ({err_ms:.2f} мс)')

plt.figure(figsize=(12, 8))

plt.subplot(2, 1, 1)
plt.plot(data_mono, label='Полный аудиосигнал')
plt.axvline(frag_start_true, color='g', linestyle=':', label=f'Истина: {frag_start_true}')
plt.axvline(est_start, color='r', linestyle='--', label=f'Оценка: {est_start}')
plt.title('Поиск фрагмента в аудио')
plt.xlabel('Отсчёт')
plt.ylabel('Амплитуда')
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(corr_audio, label='Корреляция full_audio vs fragment')
plt.axvline(est_start, color='r', linestyle='--', label='Пик корреляции')
plt.title('Корреляционная функция')
plt.xlabel('Позиция начала фрагмента')
plt.ylabel('Корреляция')
plt.legend()

plt.tight_layout()
plt.show()

if ipython_available:
    display(Audio(data_mono, rate=rate))
    display(Audio(fragment_noisy, rate=rate))
    display(Audio(data_mono[est_start:est_end], rate=rate))
else:
    print('IPython.display недоступен в текущем окружении, аудио-виджеты пропущены.')


**Ответ по разделу 6:** фрагмент WAV-записи корректно найден по максимуму корреляции; полученный индекс совпал с истинной позицией.


## 7. Рекомендации по выполнению заданий

- Для воспроизводимости результатов используйте `np.random.seed(42)` при генерации случайных чисел.
- Все графики должны быть подписаны (оси, заголовки, легенды).
- При анализе влияния шума на точность выполняйте несколько прогонов с разным `np.random.seed()` и усредняйте результаты.
- Для визуализации корреляции полезно показывать и сам сигнал, и корреляционную функцию.


## 8. Типичные ошибки и их избегание

1. **Путаница режимов `np.convolve`:** всегда проверяйте длину результата.
2. **Краевые эффекты:** при `mode='full'` корреляция даёт значения для всех возможных сдвигов, включая те, где перекрытие сигналов мало. Для поиска задержки это нормально, но при интерпретации нужно учитывать.
3. **Неправильное вычисление сдвига:** используйте `delay = argmax(corr) - (len(x)-1)`.
4. **Ненормализованное аудио:** перед прослушиванием убедитесь, что значения находятся в диапазоне [-1, 1].
5. **Сравнение с аналитикой:** при свёртке гауссовых функций не забывайте нормировать окна (сумма коэффициентов = 1), чтобы амплитуды соответствовали теоретическим.
